In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# S06 — From worked examples to an independent sports study

Read the matching lesson before running. Predictions first; source facts, manufactured controls and interpretations are labeled separately. This is not a sports performance benchmark.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
from dataclasses import asdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/shape_lab').exists())
sys.path.insert(0, str(ROOT / 'src'))
from shape_lab import sports_v8 as sport
from shape_lab.persistence import finite_bottleneck
spl, profiles = sport.load_sports(ROOT)
frames = np.array([r['frame'] for r in spl['records']])
times = sport.frame_times(frames, spl['sampling_rate'])
ball = sport.to_metres([r['ball'] for r in spl['records']], spl['xyz_unit'])
report_dir = ROOT / 'reports/v8/sports'
report_dir.mkdir(parents=True, exist_ok=True)
def save_report(stage, result):
    # Reports contain ordinary finite values; absent values are recorded explicitly.
    (report_dir / f'{stage}.json').write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Data:', len(frames), 'selected frames from one shot;', len(profiles), 'profiles from', profiles.player_id.nunique(), 'players')


Data: 12 selected frames from one shot; 12 profiles from 8 players


## Audit the required local source excerpts

In [3]:
checks=sport.verify_local_data(ROOT)
assert all(row['ok'] for row in checks)
print(json.dumps(checks,indent=2))
registry=json.loads((ROOT/'sports_v8/dataset_registry.json').read_text())
display(pd.DataFrame([{k:d[k] for k in ['name','date','status','license']} for d in registry['datasets']]))


[
  {
    "id": "spl",
    "ok": true,
    "sha256": "4ddf9c2bd6688634b1f622b7c3f8ddf6a6e5f162bf9e35bfb51e7e65c5bafdcc"
  },
  {
    "id": "skillcorner",
    "ok": true,
    "sha256": "fe4096d511e4f8811f5c504f633767ef5848c91043369d5643e57165e34303a2"
  }
]


,name,date,status,license
0,SPL Open Data free throws,Documentation March 2026; recent collection 20...,12 selected frames bundled and executed; full ...,CC BY-NC-SA 4.0
1,SkillCorner Open Data / body pose,2024/2025 A-League season; current documentati...,12 profile rows bundled; full tracking/pose un...,MIT repository license; inspect dataset-specif...
2,BasketHAR,Paper submitted 2026-04-18,Primary paper inspected; hosted data not retri...,Paper states Apache 2.0; verify actual hosted ...
3,MUVS,Zenodo v1.0 published 2026-06-15,Repository record/README inspected; video and ...,License field not resolved in rendered record;...
4,UVY / UVY-Track,Zenodo v1 published 2026-07-11,Record inspected; payload not downloaded,Provider says source videos CC BY; record lice...
5,TrackID3x3,2025 paper/repository; current README inspecte...,Official repository inspected; linked video no...,Dataset CC BY 4.0; original code Apache 2.0; t...
6,SoccerTrack v2,Paper submitted 2025-08-03,Primary paper inspected; payload not downloaded,Dataset license not verified from abstract; in...
7,SoccerNet Game State Reconstruction,Task page checked 2026-09-21; page retains his...,Official task description inspected; media not...,Review current task data/media access agreemen...
8,OpenBiomechanics,Current repository documents 2026-07-04 histor...,Current official README/license wording inspec...,Data/documentation CC BY-NC-SA 4.0 with additi...
9,BASKET-Multiview,Current official dataset page inspected 2026-0...,Official description inspected; render payload...,Inspect dataset license and any SMPL/asset res...


## Demonstrate why an LFS pointer is not a tracking dataset

In [4]:
pointer=b'version https://git-lfs.github.com/spec/v1\noid sha256:abc\nsize 97091519\n'
try:
    sport.check_payload(pointer)
except ValueError as error:
    print('Expected rejection:',error)
else:raise AssertionError('Pointer should never pass as numeric data.')


Expected rejection: Git LFS pointer received, not the dataset bytes.


## An explicit schema adapter, with a constructed—not observed—fixture

In [5]:
fixture=[{'player_id':1,'x':0.,'y':0.,'is_detected':True},{'player_id':2,'x':3.,'y':4.,'is_detected':False}]
observed,obs_ids=sport.tracking_cloud(fixture,detected_only=True)
all_provider,all_ids=sport.tracking_cloud(fixture,detected_only=False)
assert observed.shape==(1,2) and all_provider.shape==(2,2)
print('Constructed fixture:',obs_ids,all_ids)
print('No full-match tracking or body-pose payload was executed.')


Constructed fixture: [1] [1 2]
No full-match tracking or body-pose payload was executed.


## Build your own protocol before obtaining a score

In [6]:
protocol={'question':'Does a topological movement descriptor add held-out information beyond conventional features for a specified available label?', 'decision_time':'Must be declared before feature extraction', 'independent_unit':'Athlete/session/trial, chosen to match use', 'baseline':'Conventional features with training-only preprocessing', 'test_policy':'Untouched source groups', 'rights_review_required':True, 'current_excerpt_sufficient_for_prediction':False}
save_report('S06',{'local_sources':checks,'registry_entries':len(registry['datasets']),'full_source_download_executed':False,'tracking_fixture_is_manufactured':True,'protocol':protocol})
print(json.dumps(protocol,indent=2))


{
  "question": "Does a topological movement descriptor add held-out information beyond conventional features for a specified available label?",
  "decision_time": "Must be declared before feature extraction",
  "independent_unit": "Athlete/session/trial, chosen to match use",
  "baseline": "Conventional features with training-only preprocessing",
  "test_policy": "Untouched source groups",
  "rights_review_required": true,
  "current_excerpt_sufficient_for_prediction": false
}


## Independent explanation
State the observation unit, assumptions, units, one result and one unsupported conclusion. Then work the separate learner notebook without the answer notebook open.